# Checkpoint Heatmaps v2 — open_orca, K=16, 3 methods

Цель: показать визуально, что после ~step 500 per-module rank-pattern стабилизируется,
и значит warm-start в этой точке оправдан.

**Методы (все на open_orca):**
- LoRA r=16 — `logs_v2_unseeded/open_orca_lora_r16/`
- AdaLoRA init32→16 — `logs_v2/open_orca_adalora_init32_target16/`
- L1RA r=32, λ=3e-4 — `logs_v2/open_orca_l1ra_r32_lam0.0003/`

**На каждом чекпоинте (50 ... 2500 шаг каждые 50):**
1. Загружаем адаптер.
2. Для AdaLoRA / L1RA: бинпоиском τ → mean per-module survivor count = TARGET_K = 16.
3. Применяем mask → reconstructed ΔW = B · diag(gate ⊙ mask) · A (с правильной shape-конвенцией).
4. Per-(layer, module): `energy_rank_95`, `energy_rank_99`, `‖ΔW‖_F / ‖W_base‖_F`.
5. Сохраняем 3 PNG (по метрике).

**Итог:** 3 метода × 50 чекпоинтов × 3 метрики = **450 PNG**, плюс CSV summary и dynamics plot.

## 0. Colab mount (локально пропустить)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/diploma/project

## 1. Imports

In [ ]:
import os
import re
import gc
import json
import warnings
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')                 # non-interactive — быстрее
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=0.9)
print('Imports OK')

## 2. Config & paths

In [ ]:
DATASET        = 'open_orca'
TARGET_K       = 16                                # целевой mean rank (= LoRA r=16)
BASE_MODEL_ID  = 'Qwen/Qwen3-0.6B'

# Источники чекпоинтов (3 метода на open_orca)
RUN_SOURCES = {
    'lora':    Path('./logs_v2_unseeded/open_orca_lora_r16'),
    'adalora': Path('./logs_v2/open_orca_adalora_init32_target16'),
    'l1ra':    Path('./logs_v2/open_orca_l1ra_r32_lam0.0003'),
}

OUTPUT_ROOT = Path(f'./analysis_v2/qwen3_0.6b/heatmaps_{DATASET}_K{TARGET_K}')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_MODULES = {'self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj',
                  'self_attn.o_proj', 'mlp.gate_proj',  'mlp.up_proj', 'mlp.down_proj'}

MODULE_ORDER = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

print(f'Output root: {OUTPUT_ROOT.resolve()}')
for m, p in RUN_SOURCES.items():
    print(f'  {m:8s} → {p}  exists={p.exists()}')

## 3. Discover checkpoints

In [ ]:
def discover_checkpoints():
    """Список checkpoint-ов из RUN_SOURCES."""
    entries = []
    for method, run_dir in RUN_SOURCES.items():
        if not run_dir.exists():
            print(f'WARN: {run_dir} не найден, пропускаю {method}')
            continue
        ckpt_root = run_dir / 'checkpoints'
        if ckpt_root.exists():
            for sd in sorted(ckpt_root.iterdir()):
                m = re.match(r'step_(\d+)$', sd.name)
                if m and sd.is_dir():
                    entries.append({'method': method, 'step': int(m.group(1)),
                                    'path': str(sd)})
        # Финальный adapter (если есть)
        final = run_dir / 'adapter'
        if final.exists():
            entries.append({'method': method, 'step': 'final', 'path': str(final)})
    # Сортируем: внутри метода по шагу, final в конец
    entries.sort(key=lambda e: (e['method'], 999999 if e['step'] == 'final' else e['step']))
    return entries

checkpoints = discover_checkpoints()
from collections import Counter
cnts = Counter(e['method'] for e in checkpoints)
print(f'Найдено checkpoint-ов: {len(checkpoints)}')
for m, n in sorted(cnts.items()):
    print(f'  {m:8s}: {n}')

## 4. Load base model (для `‖ΔW‖_F / ‖W‖_F`)

In [ ]:
from transformers import AutoModelForCausalLM

print(f'Loading base model: {BASE_MODEL_ID} (fp32 — ~1.2GB) ...')
_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float32)
_sd   = _base.state_dict()

_BASE_KEY_RE = re.compile(r'model\.layers\.(?P<layer>\d+)\.(?P<module>\S+)\.weight$')
base_weights = {}
for key, t in _sd.items():
    m = _BASE_KEY_RE.match(key)
    if m is None:
        continue
    mod = m.group('module')
    if mod not in TARGET_MODULES:
        continue
    base_weights[(int(m.group('layer')), mod)] = t.float()

print(f'Indexed {len(base_weights)} target base weight matrices')
del _base, _sd
gc.collect()

## 5. Helpers: загрузка адаптеров, прунинг, метрики

In [ ]:
# ───────────────────── Adapter loading ─────────────────────
def _load_state_dict(adapter_dir: str) -> dict:
    p = Path(adapter_dir)
    st = p / 'adapter_model.safetensors'
    pt = p / 'adapter_model.pt'
    if st.exists():
        from safetensors.torch import load_file
        return load_file(str(st), device='cpu')
    if pt.exists():
        return torch.load(str(pt), map_location='cpu', weights_only=True)
    raise FileNotFoundError(f'Веса адаптера не найдены: {adapter_dir}')


_KEY_PATTERNS = [
    re.compile(r'layers\.(?P<layer>\d+)\.(?P<module>\S+?)\.(?P<comp>lora_[ABc]|lora_E)\.weight$'),
    re.compile(r'layers\.(?P<layer>\d+)\.(?P<module>\S+?)\.(?P<comp>lora_[ABE])$'),
    re.compile(r'layers\.(?P<layer>\d+)\.(?P<module>\S+?)\.(?P<comp>lora_[ABc])\.default$'),
]

def _parse_key(key: str):
    for pat in _KEY_PATTERNS:
        m = pat.search(key)
        if m:
            return int(m.group('layer')), m.group('module'), m.group('comp')
    return None


def collect_components(state_dict: dict) -> list[dict]:
    """Group state_dict tensors by (layer, module) and return list of
    {layer_id, module_name, A, B, gate?}.

    A/B convention is preserved as stored — calling code must know which method.
    """
    groups = {}
    for k, t in state_dict.items():
        parsed = _parse_key(k)
        if parsed is None:
            continue
        layer, module, comp = parsed
        slot = groups.setdefault((layer, module), {})
        slot[comp] = t.float()
    out = []
    for (layer, module), slot in sorted(groups.items()):
        if 'lora_A' not in slot or 'lora_B' not in slot:
            continue
        gate = None
        if 'lora_E' in slot:
            gate = slot['lora_E'].flatten()
        elif 'lora_c' in slot:
            gate = slot['lora_c'].flatten()
        out.append({'layer_id': layer, 'module_name': module,
                    'A': slot['lora_A'], 'B': slot['lora_B'], 'gate': gate})
    return out


# ───────────────────── Calibration: τ для mean K = TARGET_K ─────────────────────
def find_threshold_for_mean_k(components: list, target_k: float, max_iter: int = 60) -> dict:
    """Бинпоиск глобального порога τ на |gate|, чтобы средний per-module survivor count
    приблизился к target_k. Если у компонент нет gate (LoRA) — возвращает None.
    """
    gates = [c['gate'].abs() for c in components if c['gate'] is not None]
    if not gates:
        return None
    g_max = max(g.max().item() for g in gates)
    lo, hi = 0.0, g_max * 1.0001
    best = {'threshold': 0.0, 'mean_k': None, 'gap': float('inf')}
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        ks = [int((g >= mid).sum().item()) for g in gates]
        mean_k = float(np.mean(ks))
        gap = abs(mean_k - target_k)
        if gap < best['gap']:
            best = {'threshold': mid, 'mean_k': mean_k, 'gap': gap}
        if mean_k > target_k:
            lo = mid     # слишком много выживает → поднять порог
        else:
            hi = mid     # слишком мало → опустить
    return best


# ───────────────────── ΔW reconstruction (с прунингом по mask) ─────────────────────
def compute_pruned_delta_w(comp: dict, method: str, threshold: float | None) -> tuple[torch.Tensor, int]:
    """Вернуть (ΔW, active_rank) после применения mask = (|gate| ≥ threshold).
    Для LoRA: gate=None, threshold=None — просто ΔW = B @ A, active_rank = nominal rank.
    """
    A, B, gate = comp['A'], comp['B'], comp['gate']

    if method == 'lora':
        # PEFT: A=(r, in), B=(out, r) → ΔW = B @ A
        return (B @ A), int(A.shape[0])

    if method == 'adalora':
        # PEFT AdaLoRA: A=(r, in), B=(out, r), E=(r, 1)
        if gate is None:
            return (B @ A), int(A.shape[0])
        mask = gate.abs() >= (threshold if threshold is not None else 0.0)
        active_rank = int(mask.sum().item())
        if active_rank == 0:
            return torch.zeros(B.shape[0], A.shape[1]), 0
        gate_masked = gate.clone()
        gate_masked[~mask] = 0.0
        delta_w = B @ torch.diag(gate_masked) @ A
        return delta_w, active_rank

    if method == 'l1ra':
        # L1RA custom: A=(in, r), B=(r, out), c=(r,) → ΔW = (A * c) @ B
        if gate is None:
            return (A @ B), int(A.shape[1])
        mask = gate.abs() >= (threshold if threshold is not None else 0.0)
        active_rank = int(mask.sum().item())
        if active_rank == 0:
            return torch.zeros(A.shape[0], B.shape[1]), 0
        gate_masked = gate.clone()
        gate_masked[~mask] = 0.0
        delta_w = (A * gate_masked.view(1, -1)) @ B
        return delta_w, active_rank

    raise ValueError(f'unknown method: {method}')


# ───────────────────── Metrics ─────────────────────
def energy_rank(delta_w: torch.Tensor, threshold: float = 0.95) -> int:
    if delta_w is None or delta_w.numel() == 0:
        return 0
    try:
        sigma = torch.linalg.svdvals(delta_w)
    except Exception:
        return 0
    energy = sigma ** 2
    total  = energy.sum().item()
    if total < 1e-12:
        return 0
    cumulative = torch.cumsum(energy, dim=0) / total
    return int((cumulative >= threshold).nonzero(as_tuple=True)[0][0].item()) + 1


def short_module_name(name: str) -> str:
    return name.split('.')[-1]

print('Helpers loaded OK')

## 6. Heatmap saver

In [ ]:
def save_heatmap(rows: list[dict], metric: str, title: str, save_path: Path,
                 vmin: float = 0, vmax: float | None = None,
                 fmt: str = '.0f', cmap: str = 'YlOrRd'):
    df = pd.DataFrame(rows)
    df['module_short'] = df['module_name'].apply(short_module_name)
    pivot = df.pivot_table(index='module_short', columns='layer_id',
                            values=metric, aggfunc='first')
    pivot = pivot.reindex([m for m in MODULE_ORDER if m in pivot.index])
    if pivot.empty:
        return
    if vmax is None:
        vmax = pivot.max().max()
    fig, ax = plt.subplots(figsize=(14, 3.5))
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap,
                vmin=vmin, vmax=vmax, linewidths=0.5, ax=ax,
                annot_kws={'size': 7})
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Layer')
    ax.set_ylabel('')
    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(save_path), dpi=120, bbox_inches='tight')
    plt.close(fig)

print('save_heatmap OK')

## 7. Main loop — обработать все чекпоинты

Для каждого чекпоинта:
1. Загрузить адаптер.
2. Для AdaLoRA / L1RA — calibrate τ для mean K=TARGET_K, prune.
3. Per-(layer, module): метрики на pruned ΔW.
4. Сохранить 3 PNG (energy_rank_95, energy_rank_99, ΔW/W ratio).

In [ ]:
METRICS_CFG = [
    # (metric_key, label_for_title, vmax, fmt, cmap)
    ('energy_rank_95',    'energy_rank_95',          TARGET_K, '.0f', 'YlOrRd'),
    ('energy_rank_99',    'energy_rank_99',          TARGET_K, '.0f', 'YlOrRd'),
    ('deltaW_over_W_fro', '||ΔW||_F / ||W||_F',      None,     '.3f', 'viridis'),
]

def step_label(s):
    return 'final' if s == 'final' else f'step_{int(s):04d}'

def _all_outputs_exist(method: str, sl: str) -> bool:
    for metric, *_ in METRICS_CFG:
        if not (OUTPUT_ROOT / metric / method / f'{sl}.png').exists():
            return False
    return True

summary_rows = []
failed = []
skipped = 0
total = len(checkpoints)

for idx, ckpt in enumerate(checkpoints):
    method = ckpt['method']
    step   = ckpt['step']
    path   = ckpt['path']
    sl     = step_label(step)

    if _all_outputs_exist(method, sl):
        skipped += 1
        if (idx + 1) % 25 == 0:
            print(f'[{idx+1}/{total}] {method}/{sl}  пропуск: уже посчитано')
        continue

    print(f'[{idx+1}/{total}] {method:>8s} / {sl} ... ', end='', flush=True)
    try:
        sd = _load_state_dict(path)
        comps = collect_components(sd)
        if not comps:
            print('пропуск: LoRA-компоненты не найдены')
            continue

        # Подбор τ для AdaLoRA / L1RA
        threshold_info = None
        if method in ('adalora', 'l1ra'):
            threshold_info = find_threshold_for_mean_k(comps, target_k=float(TARGET_K))
        tau = threshold_info['threshold'] if threshold_info else None

        # Метрики по модулям после pruning ΔW
        rows = []
        for c in comps:
            dw, active_r = compute_pruned_delta_w(c, method, tau)
            er95 = energy_rank(dw, 0.95)
            er99 = energy_rank(dw, 0.99)
            dw_fro = dw.norm().item() if (dw is not None and dw.numel() > 0) else 0.0
            W = base_weights.get((c['layer_id'], c['module_name']))
            ratio = dw_fro / W.norm().item() if (W is not None and W.norm().item() > 1e-12) else 0.0
            rows.append({
                'layer_id':          c['layer_id'],
                'module_name':       c['module_name'],
                'active_rank':       active_r,
                'energy_rank_95':    er95,
                'energy_rank_99':    er99,
                'deltaW_over_W_fro': ratio,
            })

        # Сохранение heatmap
        suffix = ''
        if threshold_info is not None:
            suffix = f"  [τ={threshold_info['threshold']:.4e}, mean_K≈{threshold_info['mean_k']:.2f}]"
        for metric, label, vmax, fmt, cmap in METRICS_CFG:
            mean_val = float(np.mean([r[metric] for r in rows]))
            title = (f'{method.upper()} / {DATASET} / {sl} — {label}'
                     f'{suffix}  (mean={mean_val:.2f})')
            save_heatmap(rows, metric, title,
                          OUTPUT_ROOT / metric / method / f'{sl}.png',
                          vmin=0, vmax=vmax, fmt=fmt, cmap=cmap)

        summary_rows.append({
            'method':         method,
            'step':           step,
            'tau':            tau if tau is not None else float('nan'),
            'mean_K':         threshold_info['mean_k'] if threshold_info else float(TARGET_K),
            'mean_er95':      float(np.mean([r['energy_rank_95']    for r in rows])),
            'mean_er99':      float(np.mean([r['energy_rank_99']    for r in rows])),
            'mean_dw_ratio':  float(np.mean([r['deltaW_over_W_fro'] for r in rows])),
        })
        del sd, comps, rows
        gc.collect()
        print('OK')
    except Exception as e:
        print(f'ошибка: {e}')
        import traceback; traceback.print_exc()
        failed.append({'method': method, 'step': step, 'error': str(e)})

print('=' * 60)
print(f'Done: computed={total - skipped - len(failed)}, skipped={skipped}, failed={len(failed)}')
for f in failed:
    print(f'  {f["method"]} / {f["step"]}: {f["error"]}')

## 8. Summary CSV

In [ ]:
summary_df = pd.DataFrame(summary_rows)
summary_df['step_sort'] = summary_df['step'].apply(lambda s: 999999 if s == 'final' else int(s))
summary_df = summary_df.sort_values(['method', 'step_sort']).reset_index(drop=True)
csv_path = OUTPUT_ROOT / 'checkpoint_summary.csv'
summary_df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')
summary_df.head(10)

## 9. Dynamics overview — mean metrics over training step

Линейный график сразу даёт ответ: где конкретно ранки/нормы перестают двигаться → точка warm-start.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
metric_plots = [
    ('mean_er95',     'energy_rank_95'),
    ('mean_er99',     'energy_rank_99'),
    ('mean_dw_ratio', '||ΔW||_F / ||W||_F'),
]
plot_df = summary_df[summary_df['step'] != 'final'].copy()
plot_df['step'] = plot_df['step'].astype(int)

method_colors = {'lora': 'tab:blue', 'adalora': 'tab:orange', 'l1ra': 'tab:green'}
for ax, (col, title) in zip(axes, metric_plots):
    for method, grp in plot_df.groupby('method'):
        grp_s = grp.sort_values('step')
        ax.plot(grp_s['step'], grp_s[col], marker='o', markersize=3,
                color=method_colors.get(method, 'gray'),
                label=method, alpha=0.85)
    # Маркер точки warm-start
    ax.axvline(500, color='red', linestyle='--', alpha=0.5,
               label='warm-start trigger ≈ 500' if ax is axes[0] else None)
    ax.set_xlabel('Training step')
    ax.set_ylabel(title)
    ax.set_title(f'{DATASET} — {title} (pruned to mean K={TARGET_K})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(str(OUTPUT_ROOT / 'dynamics_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_ROOT / "dynamics_overview.png"}')

## 10. (Опционально) MP4 анимации

Из 50 PNG на метод × метрику склеиваем `step_*.png` в MP4 через `ffmpeg`.
В Colab `ffmpeg` уже есть. Раскомментируй чтобы запустить.